# **Project Notebook: RAG for personalisation**

This project shows the implementation for my MSc Computer Science project, where I investigate how using RAG can support an LLM (Gemma 4:12b as supported locally on Ollama 0.30.7) to produce fully constructed, stylistically personal sentences that accurately convey the meaning of a simple, agrammatic input prompt by retrieving relevant personal language samples from a database. 

This is all my own work, though I followed examples of code from some websites which are referenced at the appropriate points.

# **Getting set up**

See requirements.txt file

## Create language samples dataframe + check it

### Insert the name of the participant dataset you want to add as a collection

In [ ]:
import pandas as pd

# Insert CSV filename here!
df = pd.read_csv('AB123_language_samples.csv')

# Display the first few rows of the DataFrame
print(df.head())
# Display the column names and shape of the DataFrame
print(df.columns)
print(df.shape)

## Establish participant ID and messages from CSV file

In [ ]:
part_ids = df['Participant_ID'].tolist()
sample_ids = df['Sample_ID'].tolist()
documents = df['Message'].tolist()

print(part_ids[0:4])
print(sample_ids[0:4])
print(documents[0:4])

## Import embedding function & test it out

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# test on some sentences from the document list
sentences = documents[1:4]

# create one embedding for each sentence
embeddings = model.encode(sentences)

# check the shape of the embeddings
print(f'Embedding shape: {embeddings.shape}')

# look at similarities between embeddings of given sentences
similarities = model.similarity(embeddings, embeddings)

# show which are the 3 sentences being compared in the similarity table
for number, sentence in enumerate(sentences, start=1):
    print(f"Sentence {number}: {sentence}")

labels = [f"Sentence {number}" for number in range(1, len(sentences) + 1)]

# create a similarity table using pandas DataFrame
similarity_table = pd.DataFrame(
    similarities.numpy(),
    index = labels,
    columns = labels
)

print("\nSimilarity table:")
print(similarity_table.round(4))

## Import ChromaDB and set it up with embedding system

In [ ]:
# References: 
# DataCamp (2026) ‘Chroma DB tutorial: A step-by-step guide’. Available at: https://www.datacamp.com/tutorial/chromadb-tutorial-step-by-step-guide (Accessed: 12 July 2026).
# Chroma (n.d.) ‘Sentence Transformer’. Chroma Docs. Available at: https://docs.trychroma.com/integrations/embedding-models/sentence-transformer (Accessed: 12 July 2026).

import re
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction 
# Chroma embeds documents and text queries with the same
# SentenceTransformer model, keeping them in the same vector space.

# create a persistent client to store the database in a local directory
client = chromadb.PersistentClient(path = "./chroma_db")

# create an embedding function using the SentenceTransformer model
embedding_function = SentenceTransformerEmbeddingFunction(
    model_name = "all-MiniLM-L6-v2"
)

participant_collections = {}

for participant_id, participant_df in df.groupby("Participant_ID"):
    participant_key = str(participant_id)

    # Chroma collection names must use safe characters
    safe_id = re.sub(r"[^a-zA-Z0-9_-]", "_", participant_key)

    # create or get a collection for the participant
    collection = client.get_or_create_collection(
        name= f"participant_{safe_id}",
        embedding_function= embedding_function,
        metadata= {"participant_id": participant_key}
    )

    # upsert the participant's data into the collection
    collection.upsert(
        ids= participant_df["Sample_ID"].astype(str).tolist(),
        documents= participant_df["Message"].astype(str).tolist(),
        metadatas=[
            {"participant_id": participant_key}
            for _ in range(len(participant_df))
        ]
    )

    # store the collection in the dictionary for later use
    participant_collections[participant_key] = collection

    # print the number of documents in the collection for each participant
    print(
        f"Participant {participant_key}: "
        f"{collection.count()} documents in {collection.name}"
    )

### Check the new participant collection has been made

In [ ]:
collections = client.list_collections()
print(collections)

### Querying a participant’s collection for semantically similar messages

In [ ]:
# Reference: Chroma (n.d.) ‘Query and Get’. Chroma Docs. Available at: https://docs.trychroma.com/docs/querying-collections/query-and-get (Accessed: 12 July 2026).

participant_collection = participant_collections[participant_id]

result = participant_collection.query(
    query_texts=[
        "I didn't wash the car because I had 3 assignments to do"
    ],
    n_results=min(3, participant_collection.count()),
    include=["documents", "embeddings", "distances"]
)

# print the results of the query
returned_ids = result["ids"][0]
returned_documents = result["documents"][0]
returned_distances = result["distances"][0]
for rank, (record_id, document, distance) in enumerate(
    zip(returned_ids, returned_documents, returned_distances),
    start = 1):
        print(
            f"{rank}. ID: {record_id} | "
            f"Distance: {distance:.4f} | "
            f"Document: {document}"
        )

# confirm the number of documents in the collection
participant_collection.count()


# **Testing**

## Input the Participant_ID for testing

In [ ]:
# Must match the value in Participant_ID column of the CSV file
participant_id = "AB123" 

## LLM only condition

In [ ]:
# code adapted from https://www.youtube.com/watch?v=4T4Fr20yzBw
import ollama

model = "gemma4:12b"

def llm_only(prompt):
    system_prompt = f"""
    Express the input as one full, grammatical sentence, preserving its literal meaning.

    Rules:
    1. Use all the words from the input. 
    2. You may need to correct the grammar of the input.
    3. If there is '...' in the sentence, assume that this represents a missing word that needs to be filled in.
    4. Do not include comments or meanings, provide the corrected sentence only.
    5. Do not add unsupported information or change the negation or punctuation of the prompt.
    6. If the input is a question, do not answer it, just provide a grammatically correct version of the question.
    """

    response = ollama.chat(
        model = model,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        stream = False,
        think = False,
        keep_alive = "30m",
    )
    generated_response = response["message"]["content"]
    if not generated_response:
        raise ValueError(
            "The model returned an empty grammatical sentence."
        )
    return generated_response

generated_response = llm_only("no come tired...sleep")
generated_response

## LLM + RAG condition

In [ ]:
def generate_with_rag(generated_response, participant_id):
    # retrieve the most similar documents from the collection based on the prompt
    participant_key = str(participant_id)

    if participant_key not in participant_collections:
        raise ValueError(f"No collection found for participant {participant_key}")

    if not generated_response.strip():
        raise ValueError("The sentence to personalise is empty.")


    collection = participant_collections[participant_key]
    
    samples = collection.query(
        query_texts = [generated_response],
        n_results = 5,
        include = ["documents", "distances"]
    )
    
    retrieved = samples["documents"][0]

    # format retrieved documents 
    formatted_examples = "\n".join(
        f"Example {number}: {document}"
        for number, document in enumerate(retrieved, start = 1))

    # prepare the system prompt
    system_prompt = """
    Re-write the sentence so that it convincingly resembles the participant's 
    personal messaging style, using the supplied messages as evidence.

    Rules:
    1. Preserve the complete literal meaning of the sentence.
    2. Do not add facts, opinions, emotions, people, events, or implications.
    3. Do not remove information or alter negation.
    4. Do not answer or react to the sentence, only personalise it.
    5. Use the style examples only to inform wording, punctuation, emphasis and phrasing.
    6. Ensure the sentence is grammatically correct.
    7. The output should be noticeably personalised when the examples provide clear evidence of the participant's style.
    8. Return only the personalised sentence, with no explanation.
    """

    # prepare the user prompt
    user_prompt = f"""
    ## Sentence to personalise
    {generated_response}

    ## Style examples
    {formatted_examples}
    """

    # set up the chat
    RAG_response = ollama.chat(
        model = model,
        messages = [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        stream = False,
        think = False # added as Gemma was taking too long to produce output
    )
    
    personalised_sentence = RAG_response["message"]["content"].strip()
    if not personalised_sentence:
        raise ValueError(
            "The model returned an empty personalised sentence."
        )

    print("Unpersonalised sentence:", generated_response)

    return personalised_sentence, formatted_examples

generate_with_rag(generated_response, participant_id)